In [ ]:
# ============================================================
# PHYSICAL AI VISUAL DEMO
# Robot moving on top of a real room image
# Google Colab / Jupyter Notebook
# ============================================================

!pip -q install pillow

import cv2
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image as PILImage, ImageDraw
from IPython.display import Image, display


# ============================================================
# 1. UPLOAD ROOM IMAGE
# ============================================================

from google.colab import files

print("Upload room.jpg")

uploaded = files.upload()

image_path = next(iter(uploaded))


# ============================================================
# 2. READ IMAGE
# ============================================================

room = cv2.imread(image_path)

room = cv2.cvtColor(
    room,
    cv2.COLOR_BGR2RGB
)

height, width, _ = room.shape

print("Image size:", width, "x", height)


# ============================================================
# 3. DEFINE ROBOT START AND TARGET
#
# Coordinates are pixels in the room image
# ============================================================

start = np.array([
    int(width * 0.10),
    int(height * 0.82)
], dtype=float)


target = np.array([
    int(width * 0.78),
    int(height * 0.45)
], dtype=float)


# ============================================================
# 4. CREATE PATH
# ============================================================

number_of_frames = 60

trajectory = []

for t in np.linspace(
    0,
    1,
    number_of_frames
):

    # Linear interpolation

    position = (
        start * (1 - t)
        +
        target * t
    )

    # Slight curve to make motion look natural

    position[1] -= (
        80 *
        np.sin(
            np.pi * t
        )
    )

    trajectory.append(
        position
    )


# ============================================================
# 5. CREATE ANIMATION FRAMES
# ============================================================

frames = []


for frame_number, position in enumerate(trajectory):

    # Convert original image to PIL

    frame = PILImage.fromarray(
        room.copy()
    )

    draw = ImageDraw.Draw(
        frame
    )


    x = int(
        position[0]
    )

    y = int(
        position[1]
    )


    # ========================================================
    # DRAW TARGET
    # ========================================================

    tx = int(
        target[0]
    )

    ty = int(
        target[1]
    )


    draw.ellipse(
        [
            tx - 25,
            ty - 25,
            tx + 25,
            ty + 25
        ],
        outline="red",
        width=6
    )


    draw.text(
        (
            tx - 35,
            ty - 55
        ),
        "TARGET",
        fill="red"
    )


    # ========================================================
    # DRAW ROBOT BODY
    # ========================================================

    robot_width = 50
    robot_height = 60


    draw.rounded_rectangle(
        [
            x - robot_width//2,
            y - robot_height//2,
            x + robot_width//2,
            y + robot_height//2
        ],
        radius=10,
        fill="blue",
        outline="white",
        width=4
    )


    # ========================================================
    # ROBOT HEAD
    # ========================================================

    draw.rectangle(
        [
            x - 20,
            y - 55,
            x + 20,
            y - 30
        ],
        fill="lightblue",
        outline="white",
        width=3
    )


    # Eyes

    draw.ellipse(
        [
            x - 12,
            y - 48,
            x - 6,
            y - 42
        ],
        fill="black"
    )


    draw.ellipse(
        [
            x + 6,
            y - 48,
            x + 12,
            y - 42
        ],
        fill="black"
    )


    # ========================================================
    # ROBOT WHEELS
    # ========================================================

    draw.ellipse(
        [
            x - 32,
            y + 20,
            x - 18,
            y + 42
        ],
        fill="black"
    )


    draw.ellipse(
        [
            x + 18,
            y + 20,
            x + 32,
            y + 42
        ],
        fill="black"
    )


    # ========================================================
    # DRAW TRAJECTORY
    # ========================================================

    if frame_number > 1:

        path_points = [

            (
                int(p[0]),
                int(p[1])
            )

            for p in trajectory[
                :frame_number
            ]

        ]


        draw.line(
            path_points,
            fill="yellow",
            width=5
        )


    # ========================================================
    # STATUS
    # ========================================================

    draw.text(
        (
            20,
            20
        ),
        f"Physical AI Robot - Step {frame_number}",
        fill="white",
        stroke_width=2,
        stroke_fill="black"
    )


    frames.append(
        frame
    )


# ============================================================
# 6. SAVE AS ANIMATED GIF
# ============================================================

output_file = (
    "/content/"
    "physical_ai_room_robot.gif"
)


frames[0].save(

    output_file,

    save_all=True,

    append_images=frames[1:],

    duration=120,

    loop=0

)


# ============================================================
# 7. DISPLAY ANIMATION
# ============================================================

display(

    Image(
        filename=output_file
    )

)